In [1]:
import importlib

import ibis
import polars as pl
from ibis import deferred as _
from polars.testing import assert_frame_equal

import datasets
import datasets.fake
import pvm
import pvm.fields
import pvm.formulas
import pvm.pvm
from pvm.fields import Field, QuantityField, RateField
from pvm.pvm import PVM

importlib.reload(pvm.formulas)
importlib.reload(pvm.pvm)
importlib.reload(datasets)
importlib.reload(pvm)
importlib.reload(datasets.fake)
pass
pvm.__version__

'0.0.3'

In [2]:
price = RateField(
    "unit_price",
    definition=(_.volume * _.unit_price).sum() / _.volume.sum(),
    components=[
        QuantityField(
            "price_in_lc",
            definition=(_.volume * _.price_in_lc).sum() / _.volume.sum(),
        ),
        RateField(
            "fx_rate",
            definition=(_.volume * _.fx_rate * _.price_in_lc).sum() / (_.price_in_lc * _.volume).sum(),
        ),
    ],
)

revenue = Field(
    "revenue",
    definition=_.revenue.sum(),
    reconcile=False,
    components=[
        price,
        QuantityField(
            "volume",
            definition=_.volume.sum(),
        ),
        Field("flat_fee", definition=_.flat_fee.sum()),
    ],
)

cost = Field(
    "cost",
    definition=-_.cost.sum(),
    reconcile=False,
    components=[
        QuantityField(
            "cost_volume",
            definition=_.volume.sum(),
            reconcile=False,
            components=[
                RateField(
                    "yield_rate",
                    definition=_.volume.sum() / _.raw_material.sum(),
                ),
                QuantityField(
                    "raw_material",
                    definition=_.raw_material.sum(),
                ),
            ],
        ),
        RateField(
            "unit_cost",
            definition=(-_.unit_cost * _.volume).sum() / _.volume.sum(),
        ),
    ],
)

In [3]:
con = ibis.polars.connect({"sales": datasets.sales.raw})
sales = con.table("sales")

obj = PVM().set_data(sales).set_periods(_.year, ["2020", "2021"]).set_hierarchy([_.customer, _.sku])

obj.set_graph(cost)

In [4]:
assert_frame_equal(
    obj.calculate_effects().to_polars().sort("customer", "sku"),
    datasets.sales.cost_effects_by_customer_sku.sort("customer", "sku"),
    check_column_order=False,
    check_row_order=True,
    check_dtypes=False,
    atol=1e-2,
)

AssertionError: DataFrames are different (value mismatch for column 'raw_material__effect__2020')
[left]:  [-1762.0721167083118, -3418.911135503646, 5400.151353426744, 1307.5734332678535, -635.4587658099991, -307.911321317867, 3003.9157950604113, -10306.258685540404, -12076.663152977148, -152.6764353519324, -161.3661204678684, 7311.800072312348, -5272.7911321655165, -3144.945316650504, 544.8468206486681, 322.09762342020974, -9523.508392076992, 4039.6695193638125, -6897.099079542161, -868.6532103833596, 797.1405645497005, 19232.024564878524, 1793.1333962333567, -520.0527801648774, -2482.5265404370275, 117.62047942430188, 149.92590439, -3973.402773960352, -2500.6895258667405, -247.9313255008521, -887.7557438693807, -4102.307845404645, -261.47744207, -7660.269903657511, 84.5357159, 62.9305483, 2102.355474586515, -7041.0023158543245, 2948.9162991234207, -1544.2211831259422, 1248.7651605576766, 1002.5485797221227, 827.7519566412861, -7427.908008104128, -336.9664102839276, -849.4345419478557

In [5]:
unbound_table = ibis.table(sales.schema())
ibis.to_sql(obj.set_data(unbound_table).calculate_effects())

```sql
SELECT
  "t3"."customer",
  "t3"."sku",
  "t3"."cost_2020",
  "t3"."cost_volume_2020",
  "t3"."yield_rate_2020",
  "t3"."raw_material_2020",
  "t3"."unit_cost_2020",
  "t3"."cost_2021",
  "t3"."cost_volume_2021",
  "t3"."yield_rate_2021",
  "t3"."raw_material_2021",
  "t3"."unit_cost_2021",
  "t3"."cost_2021" - "t3"."cost_2020" AS "cost__change__2020",
  "t3"."cost_volume_2021" - "t3"."cost_volume_2020" AS "cost_volume__change__2020",
  "t3"."yield_rate_2021" - "t3"."yield_rate_2020" AS "yield_rate__change__2020",
  "t3"."raw_material_2021" - "t3"."raw_material_2020" AS "raw_material__change__2020",
  "t3"."unit_cost_2021" - "t3"."unit_cost_2020" AS "unit_cost__change__2020",
  CASE
    WHEN (
      COALESCE("t3"."cost_volume_2020", 0) <> 0
    )
    AND (
      COALESCE("t3"."cost_volume_2021", 0) <> 0
    )
    THEN (
      "t3"."cost_volume_2021" - "t3"."cost_volume_2020"
    ) * "t3"."unit_cost_2020" * 1.0
    ELSE (
      (
        "t3"."cost_volume_2021" * "t3"."unit_cost_2021"
      ) - (
        "t3"."cost_volume_2020" * "t3"."unit_cost_2020"
      )
    ) * 1.0
  END AS "cost_volume__effect__2020",
  CASE
    WHEN (
      COALESCE("t3"."cost_volume_2020", 0) <> 0
    )
    AND (
      COALESCE("t3"."cost_volume_2021", 0) <> 0
    )
    THEN (
      "t3"."unit_cost_2021" - "t3"."unit_cost_2020"
    ) * "t3"."cost_volume_2021" * 1.0
    ELSE 0
  END AS "unit_cost__effect__2020",
  CASE
    WHEN (
      COALESCE("t3"."raw_material_2020", 0) <> 0
    )
    AND (
      COALESCE("t3"."raw_material_2021", 0) <> 0
    )
    THEN (
      "t3"."raw_material_2021" - "t3"."raw_material_2020"
    ) * "t3"."yield_rate_2020" * CASE
      WHEN (
        COALESCE("t3"."cost_volume_2020", 0) <> 0
      )
      AND (
        COALESCE("t3"."cost_volume_2021", 0) <> 0
      )
      THEN "t3"."unit_cost_2020" * 1.0
      ELSE 1
    END
    ELSE (
      (
        "t3"."raw_material_2021" * "t3"."yield_rate_2021"
      ) - (
        "t3"."raw_material_2020" * "t3"."yield_rate_2020"
      )
    ) * CASE
      WHEN (
        COALESCE("t3"."cost_volume_2020", 0) <> 0
      )
      AND (
        COALESCE("t3"."cost_volume_2021", 0) <> 0
      )
      THEN "t3"."unit_cost_2020" * 1.0
      ELSE 1
    END
  END AS "raw_material__effect__2020",
  CASE
    WHEN (
      COALESCE("t3"."raw_material_2020", 0) <> 0
    )
    AND (
      COALESCE("t3"."raw_material_2021", 0) <> 0
    )
    THEN (
      "t3"."yield_rate_2021" - "t3"."yield_rate_2020"
    ) * "t3"."raw_material_2021" * CASE
      WHEN (
        COALESCE("t3"."cost_volume_2020", 0) <> 0
      )
      AND (
        COALESCE("t3"."cost_volume_2021", 0) <> 0
      )
      THEN "t3"."unit_cost_2020" * 1.0
      ELSE 1
    END
    ELSE 0
  END AS "yield_rate__effect__2020"
FROM (
  SELECT
    "t2"."customer",
    "t2"."sku",
    COALESCE(SUM("t2"."cost") FILTER(WHERE
      "t2"."__period__" = '2020'), 0) AS "cost_2020",
    COALESCE(SUM("t2"."cost_volume") FILTER(WHERE
      "t2"."__period__" = '2020'), 0) AS "cost_volume_2020",
    COALESCE(SUM("t2"."yield_rate") FILTER(WHERE
      "t2"."__period__" = '2020'), 0) AS "yield_rate_2020",
    COALESCE(SUM("t2"."raw_material") FILTER(WHERE
      "t2"."__period__" = '2020'), 0) AS "raw_material_2020",
    COALESCE(SUM("t2"."unit_cost") FILTER(WHERE
      "t2"."__period__" = '2020'), 0) AS "unit_cost_2020",
    COALESCE(SUM("t2"."cost") FILTER(WHERE
      "t2"."__period__" = '2021'), 0) AS "cost_2021",
    COALESCE(SUM("t2"."cost_volume") FILTER(WHERE
      "t2"."__period__" = '2021'), 0) AS "cost_volume_2021",
    COALESCE(SUM("t2"."yield_rate") FILTER(WHERE
      "t2"."__period__" = '2021'), 0) AS "yield_rate_2021",
    COALESCE(SUM("t2"."raw_material") FILTER(WHERE
      "t2"."__period__" = '2021'), 0) AS "raw_material_2021",
    COALESCE(SUM("t2"."unit_cost") FILTER(WHERE
      "t2"."__period__" = '2021'), 0) AS "unit_cost_2021"
  FROM (
    SELECT
      "t1"."customer",
      "t1"."sku",
      CAST("t1"."year" AS TEXT) AS "__period__",
      -(
        SUM("t1"."cost")
      ) AS "cost",
      SUM("t1"."volume") AS "cost_volume",
      SUM("t1"."volume") / SUM("t1"."raw_material") AS "yield_rate",
      SUM("t1"."raw_material") AS "raw_material",
      SUM(-(
        "t1"."unit_cost"
      ) * "t1"."volume") / SUM("t1"."volume") AS "unit_cost"
    FROM (
      SELECT
        *
      FROM "unbound_table_0" AS "t0"
      WHERE
        CAST("t0"."year" AS TEXT) IN ('2020', '2021')
    ) AS "t1"
    GROUP BY
      1,
      2,
      3
  ) AS "t2"
  GROUP BY
    1,
    2
) AS "t3"
```

In [ ]:
obj.calculate_effects().to_polars().select(
    "sku",
    "country",
    "unit_price__effect__2020",
    "price_in_lc__effect__2020",
    "fx_rate__effect__2020",
).with_columns(
    (pl.col("unit_price__effect__2020") - pl.col("price_in_lc__effect__2020") - pl.col("fx_rate__effect__2020")).alias(
        "diff",
    ),
).sort(["country", "sku"])

sku,country,unit_price__effect__2020,price_in_lc__effect__2020,fx_rate__effect__2020,diff
str,str,f64,f64,f64,f64
"""SKU0""","""Bouvet Island (Bouvetoya)""",-245.058235,221.171414,-465.975594,-0.254054
"""SKU1""","""Bouvet Island (Bouvetoya)""",775.666603,3633.445595,-2856.300861,-1.478131
"""SKU2""","""Bouvet Island (Bouvetoya)""",1488.422966,4013.257933,-2523.560357,-1.27461
"""SKU3""","""Bouvet Island (Bouvetoya)""",-1632.4751,2424.875364,-4055.173563,-2.176901
"""SKU4""","""Bouvet Island (Bouvetoya)""",-1822.414223,-1480.970039,-341.218744,-0.225439
…,…,…,…,…,…
"""SKU0""","""Saint Vincent and the Grenadin…",-1.24942,509.947018,-511.49295,0.296511
"""SKU1""","""Saint Vincent and the Grenadin…",-1233.187334,2225.256196,-3460.371041,1.927511
"""SKU2""","""Saint Vincent and the Grenadin…",-244.626223,1678.173782,-1923.883343,1.083338
